# Public Summary Readiness - Read Only
Run this separately; do not replace notebook 08 or modify its Job.
Makes one unauthenticated HTTPS request to api.github.com and checks whether
the secrets metadata API can be called. No secret values, tokens, names,
table reads/writes, uploads, repositories, or schedules are changed.
A successful probe is not proof of secure publishing.

In [0]:
import json
from urllib.request import Request, build_opener, HTTPRedirectHandler


class NoRedirect(HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        return None


readiness = {
    'github_api_reachable': False,
    'secret_metadata_api_callable': False,
    'secret_values_read': False,
    'authenticated_write_tested': False,
    'publication_enabled': False,
}
try:
    request = Request('https://api.github.com/',headers={'User-Agent':'TaxiLakehouseReadiness/1'})
    with build_opener(NoRedirect).open(request,timeout=10) as response:
        readiness['github_api_reachable'] = response.status == 200
except Exception:
    readiness['network_note'] = 'Request did not succeed. Do not bypass workspace restrictions.'
try:
    dbutils.secrets.listScopes()
    readiness['secret_metadata_api_callable'] = True
except Exception:
    readiness['secret_note'] = 'Supported secure credential setup still needs verification.'
print(json.dumps(readiness,indent=2))

In [0]:
print(
    dbutils.secrets.get(
        scope="nyc-taxi-publication",
        key="setup-check"
    ) == "ready"
)